<h1 style="color:#2bb2c0; text-align:center;">🎉 Добро пожаловать! Эта яцейка запускает(Устанавливает ComfyUI) 🎉</h1>

In [55]:
%%time
update = True

import os
import sys
import stat

from IPython.display import clear_output, display, HTML
# Статусы вывода
def print_status(message):
    display(HTML(f"<h2 style='color: green;'>✅ {message}</h2>"))
def print_warning(message):
    display(HTML(f"<h2 style='color: orange;'>⚠️ {message}</h2>"))
def print_error(message):
    display(HTML(f"<h2 style='color: red;'>❌ {message}</h2>"))
#!rm -rf /kaggle/working/venv
print_status("Начинаю проверку Установку(Запуск)среды выполнение дождитесь окончание...")
home_dir = '/kaggle/working'
python = '/kaggle/working/venv/bin/python'
pip = '/kaggle/working/venv/bin/pip'

def find_bin_folders(folder_path):
    bin_folders = []
    for root, dirs, files in os.walk(folder_path):
        for dir_name in dirs:
            if dir_name == 'bin':
                bin_folders.append(os.path.join(root, dir_name)) 
    return bin_folders

def installLibraries(home_dir, python, pip):
  %cd {home_dir}
  !{pip} install torch==2.7.1 torchvision==0.22.1 torchaudio==2.7.1 --index-url https://download.pytorch.org/whl/test/cu128 \
         #--index-url https://download.pytorch.org/whl/test/cu124
  !{pip} install tensorflow[and-cuda]
  !{pip} install imageio[ffmpeg] opencv-python ffmpeg-python moviepy
  !apt-get update
  !apt-get upgrade
  !apt-get install -y ffmpeg
  # TODO: download req.txt
  !wget https://q4j3.c11.e2-5.dev/downloads/req.txt
  !{pip} install -r /kaggle/working/req.txt

!pip install virtualenv

if not os.path.exists(f'{home_dir}/venv'):
    print('installing venv')
    os.chdir(home_dir)
    get_ipython().system(f'cd {home_dir}')
    
    get_ipython().system('virtualenv venv -p $(which python3.12)')
    installLibraries(home_dir, python, pip)
else:
    bin_folders = find_bin_folders('/kaggle/working/venv')
    if bin_folders:
      print("Found 'bin' folders:")
      for bin_folder in bin_folders:
        print(bin_folder)
        for filename in os.listdir(bin_folder):
            file_path = os.path.join(bin_folder, filename)
            if os.path.isfile(file_path):
                current_permissions = os.stat(file_path).st_mode
                 # Add execute permissions for the user, group, and others
                os.chmod(file_path, current_permissions | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)

if not os.path.exists(f'{home_dir}/venv/bin/python3.12'):
    get_ipython().system('cp /usr/bin/python3.12 /kaggle/working/venv/bin/')
!ln -s /kaggle/working/venv/bin/python3.12 /kaggle/working/venv/bin/python
!ln -s /kaggle/working/venv/bin/python3.12 /kaggle/working/venv/bin/python3

#Устанавливаем потдержку Kaggle моделей FP8
print_status("Дополнительные пакеты")
!pip install loguru  # или другой пакет логирования
!python -m pip install --upgrade pip
!pip install -U nvidia-ml-py
!pip install einops omegaconf
!pip install timm mediapy
!pip install diffusers==0.27.0 transformers==4.37.2
!pip install xformers

clear_output()

%cd /kaggle/working
!git clone https://github.com/Comfy-Org/ComfyUI.git
%cd ComfyUI
!git pull
!{pip} install -r requirements.txt

clear_output()
#==============================pip install plagin=============================================================================
clear_output()
print_status("Начинаю установку Нод....")

# Установка manager node
update_manager = True
%cd /kaggle/working/ComfyUI/custom_nodes
!git clone https://github.com/ltdrdata/ComfyUI-Manager.git
%cd ComfyUI-Manager

if update_manager:
    get_ipython().system('git pull')
#!{pip} install -r requirements.txt

# Установка node ComfyUI-Crystools
update_crystools = True
%cd /kaggle/working/ComfyUI/custom_nodes
!git clone https://github.com/crystian/ComfyUI-Crystools.git
%cd ComfyUI-Crystools

if update_crystools:
    get_ipython().system('git pull')
!{pip} install -r requirements.txt

# Установка node ComfyUI-GGUF
update_gguf = True
%cd /kaggle/working/ComfyUI/custom_nodes
!git clone https://github.com/city96/ComfyUI-GGUF
%cd ComfyUI-GGUF

if update_gguf:
    get_ipython().system('git pull')
!{pip} install -r requirements.txt

# Установка node Logic
update_logic = True
%cd /kaggle/working/ComfyUI/custom_nodes
!git clone https://github.com/theUpsider/ComfyUI-Logic.git
%cd ComfyUI-Logic

if update_logic:
    get_ipython().system('git pull')
!{pip} install -r requirements.txt

# Установка node image-saver
update_imagesaver = True
%cd /kaggle/working/ComfyUI/custom_nodes
!git clone https://github.com/giriss/comfy-image-saver.git
%cd comfy-image-saver

if update_imagesaver:
    get_ipython().system('git pull')
!{pip} install -r requirements.txt

# Установка node QwenVL
update_QwenVL = True
%cd /kaggle/working/ComfyUI/custom_nodes
!git clone https://github.com/1038lab/ComfyUI-QwenVL.git
%cd ComfyUI-QwenVL

if update_QwenVL:
    get_ipython().system('git pull')
!{pip} install -r requirements.txt

clear_output()
print_status("Установка Нод завершена успешно!..")
#==============================install Logic MULTI-GPU============================================================
# Pinggy script
!wget https://raw.githubusercontent.com/wandaweb/jupyter-webui-tunneling/main/pinggy.py -O /kaggle/working/pinggy.py  
# Second GPU offload
%cd /kaggle/working/ComfyUI/custom_nodes
!wget https://gist.githubusercontent.com/city96/30743dfdfe129b331b5676a79c3a8a39/raw/ecb4f6f5202c20ea723186c93da308212ba04cfb/ComfyBootlegOffload.py
clear_output()
print_status("✅Все готова копирую модели в нужные папки....")

#========================instal Model=============================================================================
#========================Модели Flux==============================================================================
!ln -s /kaggle/input/datasets/theangel/flux2-dev32b/flux2-dev-Q4_0.gguf \
      /kaggle/working/ComfyUI/models/diffusion_models/flux2-dev-Q4_0.gguf

!ln -s /kaggle/input/datasets/theangel/flux2-dev32b/mistral_3_small_flux2_fp8.safetensors \
      /kaggle/working/ComfyUI/models/text_encoders/mistral_3_small_flux2_fp8.safetensors

!ln -s /kaggle/input/datasets/theangel/flux2-dev32b/flux2-vae.safetensors \
      /kaggle/working/ComfyUI/models/vae/flux2-vae.safetensors
#========================Lora Flux====================================================================================
#===================Проверка Трайтон==================================================================================
# Ждем пока обновит Kaggle структуру CUDA
#=====================================================================================================================
clear_output()
print_status("Все готова мозно запускать ComfyUI")

CPU times: user 406 ms, sys: 385 ms, total: 790 ms
Wall time: 24.8 s


<h1 style="color:#2bc039; text-align:center;">🎉 Запуск ComfyUI 🎉</h1>

In [57]:
# ======================================================
# COMFYUI + CLOUDFLARE API MODE FOR HERMES AGENT
# ======================================================

import os
import time
import socket
import subprocess
import re
import select
import signal
from threading import Thread
from IPython.display import display, HTML

# ======================================================
# UI HELPERS
# ======================================================

def print_status(message):
    display(HTML(f"<h3 style='color:green;'>✅ {message}</h3>"))

def print_warning(message):
    display(HTML(f"<h3 style='color:orange;'>⚠️ {message}</h3>"))

def print_error(message):
    display(HTML(f"<h3 style='color:red;'>❌ {message}</h3>"))

# ======================================================
# PATHS
# ======================================================

COMFY_PATH = "/kaggle/working/ComfyUI"
VENV_PYTHON = "/kaggle/working/venv/bin/python"
PORT = 8188

# ======================================================
# КРИТИЧЕСКИ ВАЖНО: УБИВАЕМ СТАРЫЕ ПРОЦЕССЫ
# ======================================================

print_status("Очистка старых процессов...")

# Убиваем все старые процессы ComfyUI
subprocess.run(["pkill", "-9", "-f", "main.py"], capture_output=True)
subprocess.run(["pkill", "-9", "-f", "comfyui"], capture_output=True)
subprocess.run(["pkill", "-9", "-f", "cloudflared"], capture_output=True)

# Ждём освобождения порта
time.sleep(3)

# Удаляем файл блокировки БД
db_path = f"{COMFY_PATH}/user/comfyui.db"
db_lock = f"{COMFY_PATH}/user/comfyui.db-journal"
for f in [db_path, db_lock]:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Удалён файл блокировки: {f}")
        except:
            pass

# Проверяем что порт свободен
try:
    sock = socket.socket()
    sock.settimeout(1)
    sock.connect(("127.0.0.1", PORT))
    sock.close()
    print_warning(f"Порт {PORT} всё ещё занят! Ждём ещё 5 секунд...")
    time.sleep(5)
except:
    pass

print_status("Старые процессы очищены")

# ======================================================
# CHECK FILES
# ======================================================

if not os.path.exists(COMFY_PATH):
    raise RuntimeError(f"ComfyUI not found: {COMFY_PATH}")

if not os.path.exists(f"{COMFY_PATH}/main.py"):
    raise RuntimeError("main.py not found")

if not os.path.exists(VENV_PYTHON):
    raise RuntimeError(f"Python venv not found: {VENV_PYTHON}")

print_status("ComfyUI найден")

# ======================================================
# SHOW TORCH VERSION
# ======================================================

try:
    result = subprocess.check_output(
        [
            VENV_PYTHON,
            "-c",
            "import torch; print('Torch:', torch.__version__); print('CUDA:', torch.version.cuda)"
        ],
        text=True
    )
    print(result)
except Exception as e:
    print_warning(f"Не удалось определить Torch: {e}")

# ======================================================
# INSTALL CLOUDFLARED
# ======================================================

print_status("Установка Cloudflared")

os.chdir("/kaggle/working")

if not os.path.exists("/kaggle/working/cloudflared"):
    subprocess.run([
        "wget",
        "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O",
        "cloudflared"
    ])

    subprocess.run([
        "chmod",
        "+x",
        "cloudflared"
    ])

print_status("Cloudflared готов")

# ======================================================
# LOG STREAMER
# ======================================================

def stream_logs(proc, prefix=""):
    while True:
        try:
            line = proc.stdout.readline()

            if line:
                print(f"{prefix}{line.rstrip()}")

            elif proc.poll() is not None:
                break

            time.sleep(0.05)
        except:
            break

# ======================================================
# START COMFYUI
# ======================================================

print_status("Запуск ComfyUI")

comfy_process = subprocess.Popen(
    [
        VENV_PYTHON,
        "main.py",
        "--listen",
        "0.0.0.0",
        "--port",
        str(PORT),
        "--enable-cors-header",
        "*",
        "--disable-auto-launch",
        "--disable-smart-memory",
        "--preview-method",
        "auto"
    ],
    cwd=COMFY_PATH,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

Thread(
    target=stream_logs,
    args=(comfy_process, "[COMFY] "),
    daemon=True
).start()

# ======================================================
# WAIT FOR PORT
# ======================================================

print_status("Ожидание запуска сервера")

startup_timeout = 180
start_time = time.time()

while True:

    if comfy_process.poll() is not None:
        # Читаем оставшиеся логи чтобы увидеть ошибку
        remaining = comfy_process.stdout.read()
        if remaining:
            print(f"[COMFY] {remaining}")
        raise RuntimeError(
            f"ComfyUI завершился с кодом {comfy_process.returncode}"
        )

    try:
        sock = socket.socket()
        sock.settimeout(2)
        sock.connect(("127.0.0.1", PORT))
        sock.close()
        break

    except:
        pass

    if time.time() - start_time > startup_timeout:
        raise RuntimeError(
            "Таймаут запуска ComfyUI (180 секунд)"
        )

    time.sleep(2)

print_status("ComfyUI успешно запущен")

# ======================================================
# START CLOUDFLARE
# ======================================================

print_status("Запуск Cloudflare Tunnel")

tunnel_process = subprocess.Popen(
    [
        "/kaggle/working/cloudflared",
        "tunnel",
        "--no-autoupdate",
        "--protocol",
        "http2",
        "--url",
        f"http://127.0.0.1:{PORT}"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

# ======================================================
# WAIT FOR URL (читаем в основном потоке, чтобы избежать гонки)
# ======================================================

print_status("Ожидание ссылки Cloudflare")

public_url = None
start_time = time.time()

while time.time() - start_time < 60:
    if tunnel_process.poll() is not None:
        print_error("Tunnel процесс завершился")
        break
    
    try:
        ready, _, _ = select.select([tunnel_process.stdout], [], [], 1.0)
        if ready:
            line = tunnel_process.stdout.readline()
            if line:
                print(f"[TUNNEL] {line.rstrip()}")
                if "trycloudflare.com" in line:
                    match = re.search(
                        r"https://[^\s]+trycloudflare\.com",
                        line
                    )
                    if match:
                        public_url = match.group(0)
                        break
    except Exception as e:
        print_warning(f"Ошибка чтения: {e}")
        break

# Запускаем фоновый поток для оставшихся логов
Thread(
    target=stream_logs,
    args=(tunnel_process, "[TUNNEL] "),
    daemon=True
).start()

# ======================================================
# RESULT
# ======================================================

if public_url:
    print_status("ComfyUI доступен")
    print("\n" + "=" * 70)
    print("COMFYUI URL:")
    print(public_url)
    print("=" * 70 + "\n")
    
    display(HTML(f"""
    <div style='background:#f0f8ff; padding:20px; border-radius:10px; text-align:center; border:2px solid #3498db;'>
        <h2 style='color:#2c3e50;'>🐰 ComfyUI готов к работе!</h2>
        <p style='font-size:16px; color:#555;'>Нажмите кнопку ниже, чтобы открыть интерфейс в новой вкладке:</p>
        <a href='{public_url}' target='_blank' rel='noopener noreferrer'
           style='background:#3498db; color:white; padding:15px 30px; 
                  text-decoration:none; border-radius:8px; font-size:16px;
                  display:inline-block; margin-top:10px; font-weight:bold;
                  box-shadow:0 4px 6px rgba(0,0,0,0.1);'>
           🚀 Открыть ComfyUI
        </a>
        <p style='font-size:12px; color:#888; margin-top:15px;'>
            Ссылка: <code>{public_url}</code>
        </p>
        <p style='font-size:11px; color:#999; margin-top:10px;'>
            Если видите 403 ошибку, попробуйте обновить страницу (Ctrl+R) или открыть в режиме инкогнито
        </p>
    </div>
    """))
else:
    print_warning(
        "Tunnel запущен, но ссылка автоматически не обнаружена в логах."
    )
    print_warning("Проверьте логи выше — возможно, URL выводится в другом формате.")

print_status("Готово")

Удалён файл блокировки: /kaggle/working/ComfyUI/user/comfyui.db


/kaggle/working/venv/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Torch: 2.7.1+cu128
CUDA: 12.8



[COMFY] [INFO] setup plugin alembic.autogenerate.schemas
[COMFY] [INFO] setup plugin alembic.autogenerate.tables
[COMFY] [INFO] setup plugin alembic.autogenerate.types
[COMFY] [INFO] setup plugin alembic.autogenerate.constraints
[COMFY] [INFO] setup plugin alembic.autogenerate.defaults
[COMFY] [INFO] setup plugin alembic.autogenerate.comments
[COMFY] [INFO] [ComfyUI-Manager] Using `uv` as Python module for pip operations.
[COMFY] Using Python 3.12.13 environment at: /usr
[COMFY] [START] Security scan
[COMFY] [DONE] Security scan
[COMFY] ## ComfyUI-Manager: installing dependencies done.
[COMFY] ** ComfyUI startup time: 2026-06-05 11:34:38.911
[COMFY] ** Platform: Linux
[COMFY] ** Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
[COMFY] ** Python executable: /kaggle/working/venv/bin/python
[COMFY] ** ComfyUI Path: /kaggle/working/ComfyUI
[COMFY] ** ComfyUI Base Folder Path: /kaggle/working/ComfyUI
[COMFY] ** User directory: /kaggle/working/ComfyUI/user
[COMFY] ** ComfyU

[COMFY] [INFO]    0.1 seconds: /kaggle/working/ComfyUI/custom_nodes/ComfyUI-Crystools
[COMFY] [INFO]    0.1 seconds: /kaggle/working/ComfyUI/custom_nodes/ComfyUI-Manager
[COMFY] [INFO]    0.2 seconds: /kaggle/working/ComfyUI/custom_nodes/ComfyUI-QwenVL
[TUNNEL] 2026-06-05T11:34:49Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
[COMFY] [INFO]
[COMFY] [INFO] Context impl SQLiteImpl.
[COMFY] [INFO] Will assume non-transactional DDL.
[COMFY] [INFO] Context impl SQL


COMFYUI URL:
https://keyboard-obtain-exchange-gentle.trycloudflare.com



[TUNNEL] 2026-06-05T11:34:52Z INF Cannot determine default configuration path. No file [config.yml config.yaml] in [~/.cloudflared ~/.cloudflare-warp ~/cloudflare-warp /etc/cloudflared /usr/local/etc/cloudflared]
[TUNNEL] 2026-06-05T11:34:52Z INF Version 2026.5.2 (Checksum 5286698547f03df745adb2355f04c12dde52ef425491e81f433642d695521886)
[TUNNEL] 2026-06-05T11:34:52Z INF GOOS: linux, GOVersion: go1.26.3, GoArch: amd64
[TUNNEL] 2026-06-05T11:34:52Z INF Settings: map[ha-connections:1 no-autoupdate:true p:http2 protocol:http2 url:http://127.0.0.1:8188]
[TUNNEL] 2026-06-05T11:34:52Z INF Generated Connector ID: f7a50db6-bf85-4131-b14a-0c57ae83e398
[TUNNEL] 2026-06-05T11:34:52Z INF Initial protocol http2
[TUNNEL] 2026-06-05T11:34:52Z INF ICMP proxy will use 172.19.2.2 as source for IPv4
[TUNNEL] 2026-06-05T11:34:52Z INF ICMP proxy will use ::1 in zone lo as source for IPv6
[TUNNEL] 2026/06/05 11:34:52 failed to sufficiently increase receive buffer size (was: 208 kiB, wanted: 7168 kiB, got: 4